# 聚类 Bootstrap（OHIE 实验）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_student.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对 treatment 效应按家庭聚类做200次 Bootstrap，并手工计算百分位法置信区间。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all` 后载入 OHIE 实验数据（含学生出生年份、是否接受 treatment、家庭编号 household_id）。


In [ ]:
%%stata
clear all
use ../datasets/OHIE_QJE.dta


**bootstrap**

`bootstrap b=_b[treatment] se=_se[treatment], reps(200) seed(55) cluster(household_id) saving(bootstrap_student.dta, replace): reg birthyear_list treatment`：**聚类 Bootstrap**。要点：
- `b=_b[treatment]` 让 bootstrap 根据重抽样样本重新执行回归并收集 treatment 系数；`se=_se[treatment]` 同时收集标准误；
- `reps(200)` 重复200次、`seed(55)` 固定种子；
- `cluster(household_id)` 是**聚类抽样**：同一家庭的学生被整组重抽（因为同家庭个体的误差项相关），这是处理聚类相关数据时正确的 bootstrap 做法；
- `saving()` 把200组系数与标准误存成数据集备用。


In [ ]:
%%stata
bootstrap b=_b[treatment] se=_se[treatment], reps(200) seed(55) cluster(household_id) saving(bootstrap_student.dta, replace): reg birthyear_list treatment, vce(bootstrap)


**进行回归，保存系数和标准误**

先 `reg birthyear_list treatment` 得到原始样本的点估计 `b` 与标准误 `se`（后面要用它们构造置信区间）。


In [ ]:
%%stata
reg birthyear_list treatment
local b=_b[treatment]
local se=_se[treatment]


**打开一个数据框，装入bootstrap_student.dta**

打开保存的 bootstrap 结果并在其基础上构造**百分位法置信区间**：`frame create/change` 切到新数据框，`use bootstrap_student.dta` 载入结果；`gen t=(b-b)/se` 构造每个重抽样本的 t 统计量；排序后取 2.5% 与 97.5% 两个分位数 `alpha_25`、`alpha_975`（取相邻两个次序统计量的平均以避免取整误差）。


In [ ]:
%%stata
frame create bootstrap_sample
frame change bootstrap_sample
use bootstrap_student.dta
gen t=(b-`b')/se
sort t
local alpha_25=(t[floor(_N*0.025)]+t[ceil(_N*0.025)])/2
local alpha_975=(t[floor(_N*0.975)]+t[ceil(_N*0.975)])/2


**计算好分位数，返回**

`frame change default` 切回主数据框，`rm bootstrap_student.dta` 删除中间文件（避免污染工作目录）。


In [ ]:
%%stata
frame change default
rm bootstrap_student.dta


**计算置信区间**

用分位数构造并输出**有效置信区间**：`[b-se×alpha_975, b-se×alpha_25]`。由于 t 统计量的分布分位数（而不是 ±1.96）被用来缩放，这种方法能自动校正有限样本下系数分布的偏度。


In [ ]:
%%stata
di "置信区间为：" _skip "[" `b'-`alpha_975'*`se' "," `b'-`alpha_25'*`se' "]"
